# Part 2: Organisation "NotTeams" Messages

---

### Install Python packages (pip only)

In [2]:
%pip install networkx matplotlib numpy json

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement json (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for json


### Import Python packages

In [3]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import json

---

##### Examine the file "chat_data.edgelist" which represents instant messaging behaviour at an organisation. Each line contains two numbers, 𝑢 and 𝑣, separated by a blank space. Consider each number as an identifier for an individual in an organisation, with the space on each line representing that the individual, 𝑢, sent at least one message to another individual, 𝑣, at some point. Assume that messages can only sent between pairs of users and therefore group messages are not possible.

##### Additionally, examine the JSON file "chat_data_departments.json" (departments file). Keys in the departments file represent individuals using the same ids as in the "chat_data.edgelist" file and the values represent a department id that the individual can be attributed to.

##### Model the data using an appropriate, directed network representation and answer the following questions:

##### Q1. Could the connectivity within the network's largest, strongly connected component be suggested to be reflective of a small world phenomenon in comparison to the typical connectivity of 10 comparative random networks?

In [4]:
# Load the edgelist as a directed network
G = nx.read_edgelist("chat_data.edgelist", create_using=nx.DiGraph())

# Extract the largest strongly connected component (LSCC) - from Reddit/GitHub notebooks
strongly_connected_components = nx.strongly_connected_components(G)
strongly_connected_components_sorted = sorted(strongly_connected_components, key=len, reverse=True)
LSCC = G.subgraph(strongly_connected_components_sorted[0]).copy()

print(f"Number of nodes in G: {G.number_of_nodes()}")
print(f"Number of nodes in LSCC: {LSCC.number_of_nodes()}")
print(f"Number of edges in LSCC: {LSCC.number_of_edges()}")

# Calculate small world metrics for the LSCC
lscc_avg_sp = nx.average_shortest_path_length(LSCC)
lscc_avg_cc  = nx.average_clustering(LSCC)

print(f"\nLSCC Average shortest path length: {lscc_avg_sp:.2f}")
print(f"LSCC Average clustering coefficient: {lscc_avg_cc:.2f}")

# Compare against 10 random networks with same number of nodes and edges - from Twitch/Reddit notebooks
random_avg_sp_list = []
random_avg_cc_list = []

for i in range(10):
    R = nx.gnm_random_graph(LSCC.number_of_nodes(), LSCC.number_of_edges(), seed=i, directed=True)
    # Only calculate if strongly connected
    if nx.is_strongly_connected(R):
        random_avg_sp_list.append(nx.average_shortest_path_length(R))
    random_avg_cc_list.append(nx.average_clustering(R))

print("Random networks (up to 10):")
print(f"Mean average shortest path length: {np.mean(random_avg_sp_list):.2f}")
print(f"Mean average clustering coefficient: {np.mean(random_avg_cc_list):.2f}")


Number of nodes in G: 817
Number of nodes in LSCC: 746
Number of edges in LSCC: 13298

LSCC Average shortest path length: 2.84
LSCC Average clustering coefficient: 0.23
Random networks (up to 10):
Mean average shortest path length: 2.62
Mean average clustering coefficient: 0.02


##### Q2. How many individuals in the network have a higher or lower ratio of mutual connections than the ratio of mutual connections found in the overall network?

In [5]:
# Calculate overall network reciprocity
overall_reciprocity = nx.overall_reciprocity(G)
print(f"Overall network reciprocity: {overall_reciprocity:.2f}")

# Calculate per-node reciprocity
reciprocity_per_node = nx.reciprocity(G, G.nodes())
reciprocity_values = list(reciprocity_per_node.values())

higher = sum(1 for v in reciprocity_values if v > overall_reciprocity)
lower  = sum(1 for v in reciprocity_values if v < overall_reciprocity)

print(f"\nIndividuals with reciprocity higher than overall: {higher}")
print(f"Individuals with reciprocity lower than overall:  {lower}")

Overall network reciprocity: 0.49

Individuals with reciprocity higher than overall: 332
Individuals with reciprocity lower than overall:  485


##### Q3. Are occurrences of induced, connected subgraphs of 3 individuals (triads) with only mutual connections more abundant in the network than those with a mixture of mutual and asymmetric edges?



In [6]:
tc = nx.triadic_census(G)

# Only mutual connections: 201 (two mutual edges) and 300 (three mutual edges)
only_mutual = tc["201"] + tc["300"]

# Mix of mutual and asymmetric edges: 111D, 111U, 120D, 120U, 120C, 210
mixed = tc["111D"] + tc["111U"] + tc["120D"] + tc["120U"] + tc["120C"] + tc["210"]

print(f"Only mutual triads (201 + 300):                        {only_mutual}")
print(f"Mixed mutual+asymmetric triads (111D/U+120D/U/C+210): {mixed}")
print(f"\nOnly mutual > mixed: {only_mutual > mixed}")

Only mutual triads (201 + 300):                        53709
Mixed mutual+asymmetric triads (111D/U+120D/U/C+210): 214273

Only mutual > mixed: False


##### Q4. How many individuals only send emails within their department?

In [7]:
# Load department data
with open("chat_data_departments.json") as f:
    departments = json.load(f)

nodeOrder = list(G.nodes())

count = 0
for node in nodeOrder:
    out_neighbours = list(G.successors(node))
    if len(out_neighbours) == 0:
        continue
    node_dept = departments.get(node)
    # Check every message target is in the same department
    if all(departments.get(neighbour) == node_dept for neighbour in out_neighbours):
        count += 1

print(f"Individuals who only send messages within their own department: {count}")


Individuals who only send messages within their own department: 17


##### Q5. Are all departments with 10 or more members more tightly connected amongst themselves in comparison to all individuals across the overall network irrespective of their department?  Where in this context, 'more tightly connected' is defined as having less connection sparsity and more reciprocated connections. In addition to answering the overall question as yes or no, provide a list of departments this is true for (if any) and not true for (if any).

In [9]:
# Overall network density and reciprocity for comparison - from GitHub/Reddit notebooks
overall_density = nx.density(G)
overall_reciprocity = nx.overall_reciprocity(G)

print(f"Overall network density:     {overall_density:.2f}")
print(f"Overall network reciprocity: {overall_reciprocity:.2f}\n")

# Group nodes by department
dept_members = {}
for node, dept in departments.items():
    if node in G:
        dept_members.setdefault(dept, []).append(node)

# Filter to departments with 10 or more members
large_depts = {d: members for d, members in dept_members.items() if len(members) >= 10}
print(f"Departments with 10+ members: {len(large_depts)}\n")

more_tightly = []
not_more_tightly = []

for dept, members in sorted(large_depts.items()):
    subgraph = G.subgraph(members).copy()  # induced subgraph - from GitHub commits notebook
    dept_density = nx.density(subgraph)
    dept_reciprocity = nx.overall_reciprocity(subgraph) if subgraph.number_of_edges() > 0 else 0.0

    tighter = dept_density > overall_density and dept_reciprocity > overall_reciprocity

    print(f"Dept {dept}: members={len(members)}, density={dept_density:.2f}, reciprocity={dept_reciprocity:.2f}, tighter={tighter}")

    if tighter:
        more_tightly.append(dept)
    else:
        not_more_tightly.append(dept)

more_tightly.sort(key=int)
not_more_tightly.sort(key=int)

print(f"\nDepartments more tightly connected: {more_tightly}")
print(f"Departments NOT more tightly connected: {not_more_tightly}")
print(f"\nAll departments more tightly connected: {len(not_more_tightly) == 0}")

Overall network density:     0.02
Overall network reciprocity: 0.49

Departments with 10+ members: 21

Dept 1: members=50, density=0.03, reciprocity=0.46, tighter=False
Dept 11: members=25, density=0.01, reciprocity=0.40, tighter=False
Dept 13: members=23, density=0.03, reciprocity=0.80, tighter=True
Dept 14: members=80, density=0.02, reciprocity=0.43, tighter=False
Dept 15: members=50, density=0.02, reciprocity=0.47, tighter=False
Dept 16: members=16, density=0.02, reciprocity=0.40, tighter=False
Dept 17: members=26, density=0.02, reciprocity=0.57, tighter=True
Dept 19: members=157, density=0.02, reciprocity=0.51, tighter=True
Dept 20: members=11, density=0.03, reciprocity=0.67, tighter=True
Dept 22: members=19, density=0.01, reciprocity=0.40, tighter=False
Dept 23: members=18, density=0.03, reciprocity=0.25, tighter=False
Dept 3: members=12, density=0.02, reciprocity=0.67, tighter=True
Dept 34: members=11, density=0.03, reciprocity=0.67, tighter=True
Dept 36: members=21, density=0.01